# Invoke an AnthroFold Endpoint

The endpoint is up — time to send some predictions through it. This notebook reads your input CSV, splits the jobs into async batches, submits each batch to SageMaker, polls S3 for results, and saves everything locally under `outputs/invoke_<timestamp>/`.

**A few notes on inputs and limits:**

- **Inputs** are a six-column CSV (see the *Input format* section in `1-deploy-endpoint.ipynb`). See `examples/structure_determination_input.csv` for a four-row example.
- **Maximum complex size is 2048 residues total.** The notebook filters anything larger out before submitting, so oversize jobs won't silently fail mid-run.
- **Batches keep each invocation under SageMaker's 1-hour async timeout.** Default batch size is 4; lower it for large complexes if individual requests get close to the limit.
- **Outputs** are mmCIF structures plus per-prediction confidence summaries (ipTM, pLDDT, pTM, etc.).

## Input format

The notebook reads the six-column CSV format documented in `1-deploy-endpoint.ipynb`. Point `INPUT_PATH` below at your CSV. The default is `examples/structure_determination_input.csv`.

## Install dependencies

In [ ]:
%pip install -qU -r requirements.txt

## Configure

The endpoint name is picked up automatically from `endpoint_name.txt` (written by the deploy notebook). If you're targeting a different endpoint, set the `ANTHROFOLD_ENDPOINT_NAME` environment variable or edit `ENDPOINT_NAME` below.

`BATCH_SIZE` controls how many jobs ride in each async request. Keep each batch under the 1-hour async timeout (roughly 5-10 minutes per job). `MAX_RESIDUES_PER_JOB` enforces the 2048-residue model limit — oversize jobs are excluded before any S3 upload.

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.append("src")

from anthrofold_helpers import (
    AsyncInferenceFailure,
    batch_label,
    batch_manifest,
    get_sagemaker_context,
    invoke_async,
    load_jobs,
    plan_batches,
    quick_cif_plot,
    save_predictions,
    summarize_predictions,
    upload_jobs,
    wait_for_result,
    write_json,
)
from az_adapter import csv_to_jobs

SAGEMAKER_REGION = "us-east-1"
S3_BUCKET_REGION = "us-east-1"
# In SageMaker Studio, credentials come from the attached execution role; leave the
# block below commented. If running locally, point boto3 at your AWS credentials file.
# AWS_CREDS_PATH = "/path/to/.aws/credentials"
# os.environ["AWS_SHARED_CREDENTIALS_FILE"] = AWS_CREDS_PATH
# os.environ["AWS_PROFILE"] = "default"
os.environ["AWS_DEFAULT_REGION"] = SAGEMAKER_REGION
os.environ["AWS_REGION"] = SAGEMAKER_REGION

# Endpoint name is read from the sidecar file written by 1-deploy-endpoint.ipynb.
# Override by setting ANTHROFOLD_ENDPOINT_NAME or pasting the name into ENDPOINT_NAME.
ENDPOINT_NAME = os.environ.get("ANTHROFOLD_ENDPOINT_NAME", "").strip()
if not ENDPOINT_NAME:
    _sidecar = Path("endpoint_name.txt")
    if _sidecar.exists():
        ENDPOINT_NAME = _sidecar.read_text().strip()

INPUT_PATH = "examples/structure_determination_input.csv"
BATCH_SIZE = 4
MAX_RESIDUES_PER_JOB = 2048
LIMIT_BATCHES = None  # set to an integer for a smoke test

ctx = get_sagemaker_context(
    region=SAGEMAKER_REGION,
    s3_region=S3_BUCKET_REGION,
)

account_id = ctx["account_id"]
s3 = ctx["s3_client"]
runtime = ctx["runtime_client"]
customer_bucket = f"sagemaker-{S3_BUCKET_REGION}-{account_id}"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = Path("outputs") / f"invoke_{run_id}"
input_prefix = f"anthrofold/input/{run_id}/"

if not ENDPOINT_NAME or ENDPOINT_NAME == "PASTE_ENDPOINT_NAME_HERE":
    raise ValueError(
        "Endpoint name not set. Run 1-deploy-endpoint.ipynb first (which writes "
        "endpoint_name.txt), or set ANTHROFOLD_ENDPOINT_NAME, or paste the name "
        "into ENDPOINT_NAME above."
    )

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Input file: {INPUT_PATH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Run dir: {run_dir}")
print(f"S3 prefix: s3://{customer_bucket}/{input_prefix}")

## Plan batches

Jobs above the residue cap are excluded; the remaining jobs are sorted by size and split into batches. A manifest is saved alongside so you can see exactly which jobs went where (useful for resuming or re-running later).

In [ ]:
if INPUT_PATH.endswith('.csv'):
    jobs = csv_to_jobs(INPUT_PATH)
else:
    jobs = load_jobs(INPUT_PATH)
batches, excluded = plan_batches(
    jobs,
    batch_size=BATCH_SIZE,
    max_residues=MAX_RESIDUES_PER_JOB,
    sort_by_size=True,
    limit_batches=LIMIT_BATCHES,
)
manifest = batch_manifest(jobs, batches, excluded, BATCH_SIZE, MAX_RESIDUES_PER_JOB)
write_json(run_dir / "manifest.json", manifest)

print(f"Input jobs:    {manifest['input_count']}")
print(f"Selected jobs: {manifest['selected_count']}")
print(f"Excluded jobs: {manifest['excluded_count']}")
print(f"Batches:       {len(batches)}")

for item in manifest["batches"]:
    print(f"{item['batch_label']}: {item['names']} residues={item['total_residues']}")

## Submit and poll

One async invocation per batch. Each batch's input JSON, invoke response, raw result, mmCIF files, and confidence summaries are written to `run_dir` as they come back — so you can inspect partial output mid-run, or resume manually if anything goes wrong.

In [ ]:
all_saved = []
all_summaries = []
failed_batches = []

for batch_index, batch_jobs in enumerate(batches, start=1):
    label = batch_label(batch_index, batch_jobs)
    print()
    print(f"Submitting {label}")
    print([job.get("name") for job in batch_jobs])

    write_json(run_dir / "batch_inputs" / f"{label}.json", batch_jobs)
    input_s3_uri = upload_jobs(
        s3_client=s3,
        bucket=customer_bucket,
        prefix=input_prefix,
        jobs=batch_jobs,
        name=label,
    )
    print(f"Input: {input_s3_uri}")

    response = invoke_async(
        runtime_client=runtime,
        endpoint_name=ENDPOINT_NAME,
        input_s3_uri=input_s3_uri,
        invocation_timeout_seconds=3600,
        request_ttl_seconds=21600,
        output_path_extension=label,
    )
    write_json(run_dir / "invoke_responses" / f"{label}.json", response)

    output_s3_uri = response["OutputLocation"]
    failure_s3_uri = response.get("FailureLocation")
    print(f"InferenceId: {response.get('InferenceId')}")
    print(f"Output:      {output_s3_uri}")
    print(f"Failure:     {failure_s3_uri}")

    try:
        result = wait_for_result(
            s3_client=s3,
            output_s3_uri=output_s3_uri,
            failure_s3_uri=failure_s3_uri,
            poll_seconds=30,
            max_wait_seconds=4200,
        )
    except AsyncInferenceFailure as exc:
        failure_record = {
            "batch": label,
            "jobs": [job.get("name") for job in batch_jobs],
            "output_s3_uri": output_s3_uri,
            "failure_s3_uri": exc.failure_s3_uri,
            "failure_body": exc.body,
        }
        failed_batches.append(failure_record)
        write_json(run_dir / "failures" / f"{label}_failure.json", failure_record)
        print(f"Failed {label}; saved failure object to {run_dir / 'failures' / (label + '_failure.json')}")
        continue

    write_json(run_dir / "results" / f"{label}_result.json", result)

    saved = save_predictions(result, output_dir=run_dir, prefix=label)
    summaries = summarize_predictions(result)
    all_saved.extend(saved)
    all_summaries.extend({"batch": label, **row} for row in summaries)

    print(f"Finished {label}")
    for row in summaries:
        print(f"  {row['name']}: ranking_score={row['ranking_score']} iptm={row['iptm']} plddt={row['plddt']}")

write_json(run_dir / "result_manifest.json", {"saved": all_saved, "summaries": all_summaries, "failed_batches": failed_batches})
print()
print(f"Saved {len(all_saved)} predictions under {run_dir}")
print(f"Failed batches: {len(failed_batches)}")

## Inspect results

You'll get one prediction per input job. Each prediction comes with a confidence summary — here's a quick guide to the metrics you'll see most:

| Metric | Range | What it tells you |
|---|---|---|
| **ipTM** | 0-1 | Interface confidence. Higher means stronger inter-chain placement. |
| **pLDDT** | 0-100 | Mean per-residue confidence over the returned structure. |
| **pTM** | 0-1 | Global fold confidence. |
| **ranking_score** | model score | Internal score used by the model to pick the returned structure. |
| **has_clash** | bool | Flags steric clashes in the returned structure. |

In [ ]:
for row in all_summaries:
    print()
    print(f"{row['batch']} / {row['name']}")
    print(f"  ipTM:          {row['iptm']:.3f}" if isinstance(row['iptm'], (int, float)) else "  ipTM:          n/a")
    print(f"  pLDDT:         {row['plddt']:.2f}" if isinstance(row['plddt'], (int, float)) else "  pLDDT:         n/a")
    print(f"  pTM:           {row['ptm']:.3f}" if isinstance(row['ptm'], (int, float)) else "  pTM:           n/a")
    print(f"  ranking_score: {row['ranking_score']:.3f}" if isinstance(row['ranking_score'], (int, float)) else "  ranking_score: n/a")
    print(f"  has_clash:     {row['has_clash']}")
    print(f"  CIF size:      {row['cif_chars']:,} chars")

In [ ]:
# Optional inline 3D view of the first returned prediction.
if all_summaries:
    import json

    result_path = run_dir / "results" / f"{all_summaries[0]['batch']}_result.json"
    result = json.loads(result_path.read_text())
    quick_cif_plot(result["predictions"][0]["cif_content"])